# Task 1 Data Collection & Assembly
---

## FIRMS Data Ingest
This notebook provides a clean, reusable FIRMS ingestion utility.
Usage: provide a `start_date` (YYYY-MM-DD), `days` (int), and `MAP_KEY` (your FIRMS/API key).
It will attempt to download data for common FIRMS satellites (MODIS, VIIRS_SNPP, VIIRS_NOAA20) and save a single combined CSV under `data/raw/`.

Note: `base_url` may need adjustment depending on the FIRMS API access method; update it if your instructor provided a different endpoint.

## Boundary Reference

Copy one of these boxes into your FIRMS filtering code. Coordinates are `[west, south, east, north]` in `EPSG:4326`.

If you do not pass a boundary box, the notebook fetches the whole world.
If you do pass a boundary box, use it like `boundary_box=[west, south, east, north]`.

Saved file names are kept simple:
- `modis_sp_20221009_3d_turkey.csv`
- `all_20221009_3d_world.csv`
- `viirs_snpp_sp_20221009_3d_uae.csv`

**Countries**
- Turkey: `[25.7, 35.8, 45.1, 42.2]`
- Ukraine: `[22.0, 44.0, 40.0, 52.5]`
- Iran: `[44.0, 25.0, 63.5, 39.9]`
- Qatar: `[50.6, 24.4, 51.7, 26.2]`
- United Arab Emirates: `[51.5, 22.5, 56.6, 26.5]`
- Saudi Arabia: `[34.4, 16.3, 55.7, 32.2]`
- Yemen: `[42.5, 12.0, 54.6, 19.0]`
- Israel: `[34.2, 29.4, 35.9, 33.4]`
- Lebanon: `[35.1, 33.0, 36.7, 34.7]`
- Syria: `[35.7, 32.0, 42.5, 37.4]`
- Iraq: `[38.8, 29.0, 49.5, 37.4]`
- Jordan: `[34.9, 29.0, 39.4, 33.5]`
- Kuwait: `[46.5, 28.4, 48.5, 30.1]`
- Bahrain: `[50.3, 25.3, 50.9, 26.5]`
- Oman: `[51.9, 16.5, 59.9, 26.5]`
- Australia: `[112.0, -44.0, 154.0, -10.0]`

**Cities**
- Bursa: `[28.45, 40.00, 30.00, 40.55]`
- Antalya: `[29.80, 36.55, 31.25, 37.15]`
- Izmir: `[26.95, 38.10, 27.65, 38.75]`

**Regions**
- Western Russia: `[27.0, 41.0, 61.0, 56.0]`
- Black Sea Basin: `[27.0, 40.0, 46.5, 47.5]`
- Eastern Mediterranean / Levant: `[33.0, 28.0, 43.5, 37.8]`
- Red Sea Region: `[32.0, 12.0, 44.0, 28.5]`
- Bab el-Mandeb: `[42.0, 11.0, 44.8, 14.8]`
- Strait of Hormuz: `[54.0, 24.0, 57.7, 27.5]`
- Persian Gulf: `[47.0, 23.0, 57.8, 31.8]`
- Texas Permian Basin: `[-104.5, 29.0, -100.5, 34.0]`

If you want tighter boxes later, I can narrow them to the exact city or corridor.

In [1]:
import io
import time
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import requests


# --- Project root resolution ---
def _find_project_root(anchor="Final Project"):
    """Walk up from cwd until we find the 'Final Project' folder."""
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == anchor:
            return parent
    raise RuntimeError(f"Could not locate '{anchor}' directory in parents of {p}")

ROOT        = _find_project_root()
FIRMS_DIR   = ROOT / "FIRMS"
NEWS_DIR    = ROOT / "News"
GADM_DIR    = ROOT / "Boundaries" / "GADM boundaries"


KNOWN_BOUNDARY_NAMES = {
    (25.7, 35.8, 45.1, 42.2): 'turkey',
    (22.0, 44.0, 40.0, 52.5): 'ukraine',
    (44.0, 25.0, 63.5, 39.9): 'iran',
    (50.6, 24.4, 51.7, 26.2): 'qatar',
    (51.5, 22.5, 56.6, 26.5): 'uae',
    (34.4, 16.3, 55.7, 32.2): 'saudi_arabia',
    (42.5, 12.0, 54.6, 19.0): 'yemen',
    (34.2, 29.4, 35.9, 33.4): 'israel',
    (35.1, 33.0, 36.7, 34.7): 'lebanon',
    (35.7, 32.0, 42.5, 37.4): 'syria',
    (38.8, 29.0, 49.5, 37.4): 'iraq',
    (34.9, 29.0, 39.4, 33.5): 'jordan',
    (46.5, 28.4, 48.5, 30.1): 'kuwait',
    (50.3, 25.3, 50.9, 26.5): 'bahrain',
    (51.9, 16.5, 59.9, 26.5): 'oman',
    (112.0, -44.0, 154.0, -10.0): 'australia',
    (28.45, 40.0, 30.0, 40.55): 'bursa',
    (29.8, 36.55, 31.25, 37.15): 'antalya',
    (26.95, 38.1, 27.65, 38.75): 'izmir',
    (27.0, 41.0, 61.0, 56.0): 'western_russia',
    (27.0, 40.0, 46.5, 47.5): 'black_sea_basin',
    (33.0, 28.0, 43.5, 37.8): 'levant',
    (32.0, 12.0, 44.0, 28.5): 'red_sea_region',
    (42.0, 11.0, 44.8, 14.8): 'bab_el_mandeb',
    (54.0, 24.0, 57.7, 27.5): 'strait_of_hormuz',
    (47.0, 23.0, 57.8, 31.8): 'persian_gulf',
    (-104.5, 29.0, -100.5, 34.0): 'texas_permian_basin',
    (34.0, 22.0, 64.0, 40.0): 'middleeastwar',
}


def _build_acq_timestamp(frame):
    if 'acq_date' not in frame.columns:
        return frame

    acq_date = pd.to_datetime(frame['acq_date'], errors='coerce').dt.strftime('%Y-%m-%d')
    if 'acq_time' in frame.columns:
        acq_time = frame['acq_time'].astype(str).str.replace('.0', '', regex=False).str.zfill(4).str[:4]
        frame['acq_timestamp'] = pd.to_datetime(acq_date + ' ' + acq_time, errors='coerce', format='%Y-%m-%d %H%M')
    else:
        frame['acq_timestamp'] = pd.to_datetime(acq_date, errors='coerce')
    return frame


def _slugify(value):
    text = str(value).strip().lower()
    for character in (' ', '/', '\\'):
        text = text.replace(character, '_')
    return text


def _box_key(boundary_box):
    return tuple(round(float(value), 4) for value in boundary_box)


def _format_boundary_box(boundary_box):
    if boundary_box is None:
        return 'world', 'world'

    if isinstance(boundary_box, str):
        boundary_value = boundary_box.strip()
        if not boundary_value or boundary_value.lower() == 'world':
            return 'world', 'world'
        return boundary_value, _slugify(boundary_value)

    if len(boundary_box) != 4:
        raise ValueError('boundary_box must contain four values in [west, south, east, north] order')

    west, south, east, north = boundary_box
    area_value = f"{west},{south},{east},{north}"
    area_slug = KNOWN_BOUNDARY_NAMES.get(_box_key(boundary_box))
    if area_slug is None:
        area_slug = 'bbox_' + '_'.join(str(value).replace('-', 'm').replace('.', 'p') for value in boundary_box)
    return area_value, area_slug


def _build_output_filename(source_name, start_dt, total_days, area_slug):
    return f"{source_name}_{start_dt.strftime('%Y%m%d')}_{int(total_days)}d_{area_slug}.csv"


def get_global_firms_data(start_date_str, total_days, map_key, boundary_box=None, boundary_label=None):
    """Fetch FIRMS fire/thermal data and save one combined CSV under FIRMS/data/raw/."""
    start_dt = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_dt = start_dt + timedelta(days=max(int(total_days) - 1, 0))
    days_ago = (datetime.now() - start_dt).days

    if days_ago > 10:
        satellites = ['MODIS_SP', 'VIIRS_SNPP_SP', 'VIIRS_NOAA20_SP']
        print('System: Historical date detected. Using Standard Processing (_SP) streams.')
    else:
        satellites = ['MODIS_NRT', 'VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT']
        print('System: Recent date detected. Using Near Real-Time (_NRT) streams.')

    area_value, area_slug = _format_boundary_box(boundary_box)
    if boundary_label:
        area_slug = _slugify(boundary_label)
    elif area_slug == 'world':
        area_slug = 'world'

    chunks = []
    current_date = start_dt
    remaining_days = total_days

    while remaining_days > 0:
        run_days = min(remaining_days, 5)
        chunks.append({'date': current_date.strftime('%Y-%m-%d'), 'range': run_days})
        current_date += timedelta(days=run_days)
        remaining_days -= run_days

    all_dataframes = []

    for sat in satellites:
        print(f"\nLaunching stream sequence for satellite sensor: {sat}")

        for chunk in chunks:
            target_date = chunk['date']
            day_range = chunk['range']

            print(f"  -> Extracting area footprint | Area: {area_slug} | Start: {target_date} | Duration: {day_range} days")

            url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sat}/{area_value}/{day_range}/{target_date}"

            try:
                response = requests.get(url, timeout=30)

                if response.status_code == 429:
                    print('  Rate limited by NASA servers. Waiting 10 seconds before retry...')
                    time.sleep(10)
                    response = requests.get(url, timeout=30)

                if response.status_code == 200:
                    if 'Invalid MAP_KEY' in response.text:
                        print('Critical Error: The MAP_KEY provided is invalid.')
                        return None

                    df_chunk = pd.read_csv(io.StringIO(response.text))

                    if not df_chunk.empty:
                        df_chunk = _build_acq_timestamp(df_chunk)
                        df_chunk['retrieved_at'] = datetime.now(tz=None).replace(microsecond=0)
                        df_chunk['satellite'] = sat
                        df_chunk['period_start'] = start_dt.strftime('%Y-%m-%d')
                        df_chunk['period_end'] = end_dt.strftime('%Y-%m-%d')
                        df_chunk['area_label'] = area_slug
                        all_dataframes.append(df_chunk)
                    else:
                        print('     No thermal observations registered in this area window.')
                else:
                    print(f"  Fetch failed for this segment. HTTP Status: {response.status_code}")

            except Exception as e:
                print(f"  Connection issue on endpoint: {e}")

            time.sleep(1.5)

    if all_dataframes:
        master_df = pd.concat(all_dataframes, ignore_index=True)

        raw_dir = FIRMS_DIR / 'data' / 'raw'
        raw_dir.mkdir(parents=True, exist_ok=True)
        combined_file = raw_dir / _build_output_filename('all', start_dt, total_days, area_slug)
        master_df.to_csv(combined_file, index=False)

        print('\n=========================================')
        print('PIPELINE EXECUTION SUCCESSFUL')
        print(f'Total Combined Area Records: {len(master_df)}')
        print(f'Saved combined CSV to: {combined_file}')
        print('=========================================')
        return master_df
    else:
        print('\nExtraction cycle complete: No records recovered.')
        return None


In [ ]:
# --- Execution Parameters ---
MY_KEY = "4fe66d8195ccc519498954a7f8657d70"
START = "2026-01-01"  # Target starting baseline point (YYYY-MM-DD)
DURATION = 60 # Total days you want to fetch

# Optional: set a boundary box in [west, south, east, north] order.
# Leave it as None to fetch the whole world.
BOUNDARY_BOX = [50.6, 24.4, 51.7, 26.2]
# Example:
# BOUNDARY_BOX = [51.5, 22.5, 56.6, 26.5]  # UAE

# Run the function
df_world = get_global_firms_data(
    start_date_str=START,
    total_days=DURATION,
    map_key=MY_KEY,
    boundary_box=BOUNDARY_BOX,
)

# Inspect the dataset head to verify it's working
if df_world is not None:
    print(df_world.head())

## Guardian API

**Notebook overview**
- Purpose: Fetch and process Guardian articles for a target date window.
- Produces: raw JSON in `data/raw/` and cleaned CSV(s) in `data/processed/` using the schema: `date`, `news source`, `title`, `link`.
- Notes: Processing includes keyword filtering and duplicate removal.



In [ ]:
# Purpose: define a function to query The Guardian and return articles matching conflict signals
from datetime import datetime, timedelta
from typing import Optional

import pandas as pd
import requests


def get_guardian_headlines(start_date_str, total_days, api_key, country="Ukraine", return_raw=False, page_size=200, max_pages=None):
    """Fetch conflict-related articles from The Guardian Content API."""
    missile_terms = "missile OR cruise missile OR rocket OR barrage OR salvo OR attack OR attacked OR attacks OR strike OR struck OR strikes OR striked OR explosion OR exploded OR explodes OR explode OR blast OR blasted OR bomb OR bombed OR bombing OR killed OR killing OR injured OR injury OR hit OR hits OR shelling OR bombardment"
    explosion_terms = "explosion OR blast OR strike OR attack OR hit OR bombardment OR shelling"
    drone_terms = "drone OR kamikaze OR shahed OR uav"
    impact_terms = "blackout OR power outage OR emergency shutdown OR kill OR casualty OR injury OR destruction OR wildfire OR forest fire OR forestfire OR fire OR brush fire OR bushfire OR blaze OR inferno OR arson OR flames OR burning OR smoke OR spread OR spreaded OR wind OR burn OR burned"

    # Combine terms into the query string used by the Guardian API
    optional_signals = f"({missile_terms} OR {drone_terms} OR {impact_terms})"
    conflict_signals = f"({explosion_terms}) AND ({optional_signals} OR {explosion_terms})"
    query_string = f"{country} AND {conflict_signals}" if country else conflict_signals

    start_dt = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_dt = start_dt + timedelta(days=total_days - 1)
    end_date_str = end_dt.strftime("%Y-%m-%d")

    print("=" * 60)
    print(f"Query target: {country if country else 'Global'}")
    print(f"Timeframe: {start_date_str} to {end_date_str} ({total_days} day(s))")
    print(f"Paging: page_size={page_size}, max_pages={'all' if max_pages is None else max_pages}")
    print("=" * 60)

    url = "https://content.guardianapis.com/search"
    params = {
        "q": query_string,
        "from-date": start_date_str,
        "to-date": end_date_str,
        "api-key": api_key,
        "page-size": page_size,
        "order-by": "oldest",
        "show-fields": "headline,trailText",
    }

    all_rows = []
    raw_articles = []
    page = 1

    while True:
        if max_pages is not None and page > max_pages:
            break

        params["page"] = page
        try:
            response = requests.get(url, params=params, timeout=20)
            response.raise_for_status()
            payload = response.json()
            data = payload.get("response", {})

            if data.get("status") != "ok":
                print("Guardian API returned non-ok status:", data.get("status"))
                break

            results = data.get("results", [])
            if not results:
                break

            for article in results:
                fields = article.get("fields", {}) or {}
                all_rows.append(
                    {
                        "source": "The Guardian",
                        "country": country if country else "Global",
                        "date": article.get("webPublicationDate", "")[:10],
                        "published_at_utc": article.get("webPublicationDate", ""),
                        "headline": fields.get("headline") or article.get("webTitle", ""),
                        "snippet": fields.get("trailText", ""),
                        "section": article.get("sectionName", ""),
                        "url": article.get("webUrl", ""),
                        "api_id": article.get("id", ""),
                    }
                )
                raw_articles.append(
                    {
                        "query": query_string,
                        "page": page,
                        "fetched_at_utc": datetime.utcnow().isoformat() + "Z",
                        "article": article,
                    }
                )

            pages = data.get("pages", 1)
            if page >= pages:
                break
            page += 1

        except requests.RequestException as exc:
            print("Guardian request failed:", exc)
            break

    if not all_rows:
        print("No articles matched this query/time window.")
        if return_raw:
            return pd.DataFrame(), raw_articles
        return pd.DataFrame()

    df_news = pd.DataFrame(all_rows)
    df_news["date"] = pd.to_datetime(df_news["date"], errors="coerce")
    df_news = df_news.drop_duplicates(subset=["url"]).sort_values("published_at_utc").reset_index(drop=True)
    print(f"Collected {len(df_news)} articles from The Guardian.")

    if return_raw:
        return df_news, raw_articles
    return df_news


In [ ]:
# --- Configuration ---
GUARDIAN_API_KEY = "8fa62376-0756-4583-bbf4-a732a4c28c54"  # Replace with your actual key
START = "2026-01-01"
DAYS_TO_FETCH = 180
TARGET_COUNTRY = ""
PAGE_SIZE = 200
MAX_PAGES = None  # set a number like 5 if you want to cap pagination

# --- Run the Pipeline ---
df_guardian_headlines, raw_guardian_articles = get_guardian_headlines(
    start_date_str=START,
    total_days=DAYS_TO_FETCH,
    api_key=GUARDIAN_API_KEY,
    country=TARGET_COUNTRY,
    return_raw=True,
    page_size=PAGE_SIZE,
    max_pages=MAX_PAGES,
)

# --- Save raw data ---
import json

RAW_DIR = NEWS_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_JSON = RAW_DIR / f"guardian_raw_{START.replace('-', '')}_{DAYS_TO_FETCH}d.json"

with open(RAW_JSON, "w", encoding="utf-8") as f:
    json.dump(
        {
            "source": "The Guardian Open Platform",
            "query_start_date": START,
            "query_total_days": DAYS_TO_FETCH,
            "country": TARGET_COUNTRY,
            "page_size": PAGE_SIZE,
            "max_pages": "all" if MAX_PAGES is None else MAX_PAGES,
            "raw_article_count": len(raw_guardian_articles),
            "articles": raw_guardian_articles,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

# --- Inspect results in memory only ---
if df_guardian_headlines.empty:
    print("No data returned. Check API key, query strictness, or widen date range.")
else:
    print(df_guardian_headlines.head(10))
    print("\nColumns:", list(df_guardian_headlines.columns))
    print("\nDate range:", df_guardian_headlines["date"].min(), "to", df_guardian_headlines["date"].max())
    print("\nTotal rows:", len(df_guardian_headlines))

print("Saved raw JSON to:", RAW_JSON.resolve())
print("Metadata written: source, query_start_date, query_total_days, page_size, max_pages")


In [ ]:
# --- Process raw Guardian JSON -> standardized CSV(s) ---
import json
import re
import pandas as pd

ANALYSIS_KEYWORDS = [
    "missile", "explosion", "drone", "strike", "attack", "shelling",
    "bombardment", "blast", "blackout", "power outage", "infrastructure",
    "energy", "barrage", "kamikaze", "shahed", "electricity", "grid",
    "attacked", "attacks", "struck", "strikes", "striked", "exploded",
    "explodes", "explode", "blasted", "bomb", "bombed", "bombing",
    "killed", "killing", "injured", "injury", "hit", "hits", "burning",
    "burned", "fire", "wildfire", "forest fire", "forestfire", "brush fire",
    "bushfire", "blaze", "inferno", "arson", "flames", "smoke", "spread",
    "spreaded", "wind", "burn",
]

RAW_DIR  = NEWS_DIR / "data" / "raw"
PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in ANALYSIS_KEYWORDS)


def merge_headline_and_trailtext(headline, trail_text):
    headline_text = str(headline or "").strip()
    trail_text = re.sub(r"<[^>]+>", " ", str(trail_text or ""))
    trail_text = re.sub(r"\s+", " ", trail_text).strip()
    if headline_text and trail_text and trail_text.lower() != headline_text.lower():
        return f"{headline_text} - {trail_text}"
    return headline_text or trail_text


for raw_file in sorted(RAW_DIR.glob("guardian_raw_*.json")):
    try:
        obj = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to read {raw_file}: {exc}")
        continue

    articles = obj.get("articles", [])
    rows = []
    for rec in articles:
        art = rec.get("article", {})
        fields = art.get("fields", {}) or {}
        headline = fields.get("headline") or art.get("webTitle", "")
        trail_text = fields.get("trailText", "")
        title = merge_headline_and_trailtext(headline, trail_text)
        link = art.get("webUrl", "")
        if not title or not link:
            continue

        snippet = trail_text
        if not matches_keywords(f"{headline} {snippet}"):
            continue

        rows.append(
            {
                "date": parse_datetime(art.get("webPublicationDate", "")),
                "news source": "Guardian",
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching articles found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed Guardian data to: {processed_path} ({len(df)} rows)")


**What this notebook does**
- Queries The Guardian Content API for the selected date window and saves the raw API payload under `data/raw/`.
- Re-processes each saved raw file into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read the nested `article` records from the saved raw JSON.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse `webPublicationDate` into a full datetime and keep the time from the source.
- Set the news source label to `Guardian`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant articles.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
` Merging All`

In [ ]:
# Incremental merge for Guardian: keep history in guardian_all.csv and append newcomers
import pandas as pd

PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "guardian_all.csv"

guardian_files = sorted(
    f for f in PROC_DIR.glob("guardian_*.csv")
    if f.name.lower() != "guardian_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in guardian_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No Guardian data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental Guardian master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None


## NYT API

**Notebook overview**
- Purpose: Download NYT monthly archive and extract conflict-related headlines.
- Produces: raw monthly JSON under `data/raw/` and standardized CSV(s) under `data/processed/` with schema: `date`, `news source`, `title`, `link`.
- Notes: Processing includes keyword filtering and duplicate removal.



In [ ]:
# NYT: function to fetch monthly archive and filter conflict headlines
from datetime import datetime
import json
import pandas as pd
import requests

# Define your core action signal and infrastructure keywords
CONFLICT_KEYWORDS = [
    "missile", "strike", "explosion", "drone", "barrage", "blast",
    "attack", "bombardment", "shelling", "kamikaze", "shahed",
    "blackout", "power outage", "energy", "electricity", "grid",
    "infrastructure", "attacked", "attacks", "struck", "strikes", "striked",
    "exploded", "explodes", "explode", "blasted", "bomb", "bombed", "bombing",
    "killed", "killing", "injured", "injury", "hit", "hits", "burning",
    "burned", "fire", "wildfire", "forest fire", "forestfire", "brush fire",
    "bushfire", "blaze", "inferno", "arson", "flames", "smoke", "spread",
    "spreaded", "wind", "burn",
]


def get_nyt_archive_headlines(year, month, api_key, country=None):
    """Download one NYT monthly archive, filter conflict headlines, and save processed output."""
    print("==================================================")
    print(f"Accessing NYT Bulk Archive Stream for: {year}-{month:02d}")
    print("==================================================")

    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    params = {"api-key": api_key}

    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 429:
            print("Rate Limit Triggered. Please wait before running again.")
            return pd.DataFrame()
        if response.status_code != 200:
            print(f"Failed to extract archive. HTTP Status: {response.status_code}")
            return pd.DataFrame()

        data = response.json()
        all_articles = data.get("response", {}).get("docs", [])
        print(f"Successfully unpacked {len(all_articles)} total monthly articles.")

        # Save the raw API response to News/data/raw as readable JSON
        raw_dir = NEWS_DIR / "data" / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)
        raw_path = raw_dir / f"nyt_raw_{year}{month:02d}.json"
        with raw_path.open("w", encoding="utf-8") as fh:
            json.dump(data, fh, ensure_ascii=False, indent=2, default=str)
        print(f"Raw NYT archive saved: {raw_path}")

    except Exception as e:
        print(f"Network connection failed: {e}")
        return pd.DataFrame()

    filtered_headlines = []
    for doc in all_articles:
        pub_date_str = doc.get("pub_date", "")
        if not pub_date_str:
            continue
        try:
            dt = datetime.strptime(pub_date_str[:10], "%Y-%m-%d")
        except Exception:
            continue

        headline_main = doc.get("headline", {}).get("main", "")
        abstract_text = doc.get("abstract", "")
        text_pool = f"{headline_main} {abstract_text} {doc.get('lead_paragraph', '')}".lower()
        if country and country.lower() in text_pool:
            has_keyword = any(kw in text_pool for kw in CONFLICT_KEYWORDS)
            if has_keyword and headline_main:
                filtered_headlines.append(
                    {
                        "source": "The New York Times",
                        "country": country,
                        "date": dt.strftime("%Y-%m-%d"),
                        "headline": headline_main,
                        "url": doc.get("web_url", ""),
                    }
                )

    if filtered_headlines:
        df_nyt = pd.DataFrame(filtered_headlines)
        df_nyt.drop_duplicates(subset=["headline"], inplace=True)
        processed_dir = NEWS_DIR / "data" / "processed"
        processed_dir.mkdir(parents=True, exist_ok=True)
        processed_path = processed_dir / f"nyt_{year}{month:02d}.csv"
        df_nyt.to_csv(processed_path, index=False, encoding="utf-8")
        print(f"Processed NYT conflict news saved: {processed_path}")
        print(f"Slicing Success! Recovered {len(df_nyt)} clean conflict headlines.")
        return df_nyt

    print("No headlines matched your criteria within the selected month.")
    return pd.DataFrame()


In [ ]:
# --- Execution Configuration ---
MY_NYT_KEY = "8zqUF4rs33orAdv96UGfJaIi9VaYeWHAurOgFrXaPnRWMtmw"  # Paste your API key here

# Target month: October 2022
TARGET_YEAR = 2026
TARGET_MONTH = 6
COUNTRY = ""

# --- Execute Data Harvesting ---
df_nyt_final = get_nyt_archive_headlines(
    year=TARGET_YEAR,
    month=TARGET_MONTH,
    api_key=MY_NYT_KEY,
    country=COUNTRY,
)

# --- View the Collected Headline Rows ---
df_nyt_final.head()


In [ ]:
# --- Process raw NYT archive JSON(s) -> standardized CSV(s) ---
import json
import re
import pandas as pd

RAW_DIR  = NEWS_DIR / "data" / "raw"
PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

CONFLICT_KEYWORDS = [
    "missile", "strike", "explosion", "drone", "barrage", "blast",
    "attack", "bombardment", "shelling", "kamikaze", "shahed",
    "blackout", "power outage", "energy", "electricity", "grid",
    "infrastructure", "attacked", "attacks", "struck", "strikes", "striked",
    "exploded", "explodes", "explode", "blasted", "bomb", "bombed", "bombing",
    "killed", "killing", "injured", "injury", "hit", "hits", "burning",
    "burned", "fire", "wildfire", "forest fire", "forestfire", "brush fire",
    "bushfire", "blaze", "inferno", "arson", "flames", "smoke", "spread",
    "spreaded", "wind", "burn",
]


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in CONFLICT_KEYWORDS)


def combine_headline_snippet_paragraph(headline, snippet, lead_paragraph):
    headline_text = re.sub(r"\s+", " ", str(headline or "")).strip()
    snippet_text = re.sub(r"<[^>]+>", " ", str(snippet or ""))
    snippet_text = re.sub(r"\s+", " ", snippet_text).strip()
    lead_text = re.sub(r"\s+", " ", str(lead_paragraph or "")).strip()

    parts = [part for part in [headline_text, snippet_text, lead_text] if part]
    combined = " - ".join(parts)
    if combined:
        return combined
    return headline_text or snippet_text or lead_text


for raw_file in sorted(RAW_DIR.glob("nyt_raw_*.json")):
    try:
        obj = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to load {raw_file}: {exc}")
        continue

    docs = obj.get("response", {}).get("docs", [])
    rows = []
    for doc in docs:
        headline_obj = doc.get("headline")
        if isinstance(headline_obj, dict):
            headline = headline_obj.get("main", "")
        else:
            headline = headline_obj or ""
        snippet = doc.get("snippet", "")
        lead_paragraph = doc.get("lead_paragraph", "")
        title = combine_headline_snippet_paragraph(headline, snippet, lead_paragraph)
        link = doc.get("web_url", "")
        if not title or not link:
            continue

        text_pool = " ".join(
            [
                str(headline),
                str(snippet),
                str(lead_paragraph),
                str(doc.get("abstract", "")),
            ]
        )
        if not matches_keywords(text_pool):
            continue

        rows.append(
            {
                "date": parse_datetime(doc.get("pub_date", "")),
                "news source": "NYT",
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching articles found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed NYT to: {processed_path} ({len(df)} rows)")


**What this notebook does**
- Downloads the NYT monthly archive for the target month and saves the raw response under `data/raw/`.
- Re-processes each saved raw archive into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read `response.docs` from the raw JSON archive.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse `pub_date` into a full datetime and default the time to 12:00 when only a date is available.
- Set the news source label to `NYT`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant articles.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
`Merging all data`


In [ ]:
# Incremental merge for NYT: keep history in nyt_all.csv and append newcomers
import pandas as pd

PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "nyt_all.csv"

nyt_files = sorted(
    f for f in PROC_DIR.glob("nyt_*.csv")
    if f.name.lower() != "nyt_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in nyt_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No NYT data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental NYT master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None


## GoogleNews Srapping

In [ ]:
!pip install feedparser

In [ ]:
!pip install xmltojson

In [ ]:
import json
import urllib.parse
from datetime import datetime

import requests
import xmltojson


def get_raw_google_news_json(keywords, country):
    """Fetch the raw XML payload from Google News RSS, convert it to JSON, and save it to News/data/raw/."""
    raw_dir = NEWS_DIR / "data" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    executed_at = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = raw_dir / f"googlenews_raw_{executed_at}.json"

    # 1. Standard Boolean URL-encoding for the search query
    search_query = f"{country} AND ({keywords})"
    encoded_query = urllib.parse.quote(search_query)

    # Core Google News RSS Endpoint URL
    rss_url = f"https://news.google.com/rss/search?q={encoded_query}&hl=en-US&gl=US&ceid=US:en"

    print("==================================================")
    print("Requesting Raw Wire Data from Google News...")
    print(f"Target: {rss_url}")
    print("==================================================")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) CE49XProjectPipeline"
    }
    response = requests.get(rss_url, headers=headers, timeout=15)

    if response.status_code != 200:
        print(f"Network request failed with Status: {response.status_code}")
        return None

    raw_xml_string = response.text
    raw_json_string = xmltojson.parse(raw_xml_string)
    json_object = json.loads(raw_json_string)

    try:
        raw_items = json_object["rss"]["channel"]["item"]
        print(f"Success! Captured {len(raw_items)} raw, nested JSON entities.")

        print("\n--- SAMPLE RAW NESTED JSON ITEM OBJECT ---")
        print(json.dumps(raw_items[0], indent=4))
        print("------------------------------------------\n")
    except KeyError:
        print("Structure mismatch: The query might have returned 0 entries.")

    output_path.write_text(json.dumps(json_object, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Raw Google News JSON saved: {output_path}")

    return raw_json_string


In [ ]:
# --- Search Parameters ---
KEYWORDS = "missile OR explosion OR drone OR strike"
COUNTRY = "Ukraine"

# --- Execute Raw Stream Extraction ---
raw_json_output = get_raw_google_news_json(keywords=KEYWORDS, country=COUNTRY)

In [ ]:
# --- Process raw Google News JSON -> standardized CSV(s) ---
import json
import pandas as pd

ANALYSIS_KEYWORDS = [
    "missile", "explosion", "drone", "strike", "attack", "shelling",
    "bombardment", "blast", "blackout", "power outage", "infrastructure", "energy",
]

RAW_DIR  = NEWS_DIR / "data" / "raw"
PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in ANALYSIS_KEYWORDS)


for raw_file in sorted(RAW_DIR.glob("googlenews_raw_*.json")):
    try:
        obj = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to read {raw_file}: {exc}")
        continue

    items = obj.get("rss", {}).get("channel", {}).get("item", [])
    if isinstance(items, dict):
        items = [items]

    rows = []
    for item in items:
        title = item.get("title", "")
        link = item.get("link") or item.get("url")
        if not title or not link:
            continue

        source = item.get("source")
        if isinstance(source, dict):
            source = source.get("#text") or source.get("text")
        source = source or "Google News"

        if not matches_keywords(f"{title} {source}"):
            continue

        rows.append(
            {
                "date": parse_datetime(item.get("pubDate") or item.get("publication_date")),
                "news source": source,
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching items found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed Google News to: {processed_path} ({len(df)} rows)")


**What this notebook does**
- Fetches Google News RSS results for the selected query and saves the raw XML-to-JSON payload under `data/raw/`.
- Re-processes every saved raw file into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read `rss.channel.item` records from the raw JSON.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse the publication date into a full datetime; if the source only gives a date, set the time to 12:00.
- Use the RSS source name when available; otherwise fall back to `Google News`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant items.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
` Merging All News `

In [ ]:
# Incremental merge for Google News: keep history in googlenews_all.csv and append newcomers
import pandas as pd

PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "googlenews_all.csv"

gn_files = sorted(
    f for f in PROC_DIR.glob("googlenews_*.csv")
    if f.name.lower() != "googlenews_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in gn_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No Google News data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental Google News master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None


## WarZone Scrapping

**Notebook overview**
- Purpose: Scrape The War Zone for search keywords and convert scraped records into a standardized CSV for analysis.
- Produces: raw scraped JSON under `data/raw/` and cleaned CSV(s) under `data/processed/` with schema: `date`, `news source`, `title`, `link`.
- Notes: Processing parses dates, applies keyword filters, and deduplicates on `link`.



In [ ]:
import json
import re
import time
import urllib.parse
from bs4 import BeautifulSoup
import requests

# Scraper: loops keywords, fetches search results, and extracts article fields
def scrape_news_by_keywords(keywords):
    """Loops through a list of search keywords, extracts search results, cleans formatting, removes duplicate articles, and returns a structured JSON array."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) ProjectDataPipeline"
    }

    # Dictionary to prevent duplicate articles across different keyword searches
    unique_articles = {}

    print("==================================================")
    print(f"Starting news extraction for {len(keywords)} keywords...")
    print("==================================================")

    for keyword in keywords:
        encoded_query = urllib.parse.quote(keyword)
        url = f"https://www.twz.com/?s={encoded_query}"

        print(f"Searching: '{keyword}' -> {url}")

        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code != 200:
                print(f"   Skipped: HTTP Status {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, "html.parser")
            articles = soup.find_all("div", class_="post-content")
            new_items_found = 0

            for article in articles:
                # Internal helper to clean text fields
                def clean_text(element):
                    if not element:
                        return None
                    return re.sub(r"\s+", " ", element.text.strip())

                # Extract individual fields
                h3_tag = article.find("h3", class_="card-post-title")
                title = clean_text(h3_tag)

                date_tag = article.find("p", class_="byline-item-timestamp")
                date = clean_text(date_tag)
                if date:
                    date = (
                        date.replace("Posted on ", "")
                        .replace("Updated on ", "")
                        .strip()
                    )

                author_tag = article.find("a", class_="byline-link")
                author = clean_text(author_tag)

                category_tag = article.find("a", class_="cat-name-badge")
                category = clean_text(category_tag)

                link_tag = article.find(
                    "a", class_="card-post-title-link"
                ) or article.find("a")
                article_url = link_tag.get("href") if link_tag else None

                if article_url:
                    # Resolve relative paths to full URLs if necessary
                    if article_url.startswith("/"):
                        article_url = f"https://www.twz.com{article_url}"

                    # Add to dictionary if this article URL has not been processed yet
                    if article_url not in unique_articles:
                        unique_articles[article_url] = {
                            "title": title,
                            "date": date,
                            "author": author,
                            "category": category,
                            "url": article_url,
                        }
                        new_items_found += 1

            print(f"   Added {new_items_found} new unique articles.")
            time.sleep(1.5)

        except Exception as error:
            print(f"   Error processing keyword '{keyword}': {error}")

    # Convert the dictionary values into a direct list
    articles_list = list(unique_articles.values())

    # Format the final clean list as a JSON array string
    json_output = json.dumps(articles_list, indent=4)

    print("\n=========================================")
    print("EXTRACTION COMPLETE")
    print(f"Total Unique Articles Saved: {len(articles_list)}")
    print("=========================================")

    return json_output


In [ ]:
from datetime import datetime

# Define your search keywords directly
search_terms = [
    "ukraine missile",
    "shahed drone",
    "black sea fleet"
    ]

# Run the clean pipeline
scraped_data_json = scrape_news_by_keywords(search_terms)

# Preview the clean output string
if scraped_data_json:
    print(scraped_data_json[:1000])

    # Save raw output into News/data/raw/ with timestamped filename
    raw_dir = NEWS_DIR / "data" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    executed_at = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = raw_dir / f"warzone_raw_{executed_at}.json"
    output_path.write_text(scraped_data_json, encoding="utf-8")
    print(f"Raw WarZone JSON saved: {output_path}")


In [ ]:
# --- Process raw WarZone JSON -> standardized CSV(s) ---
import json
import pandas as pd

ANALYSIS_KEYWORDS = [
    "ukraine", "missile", "explosion", "drone", "strike", "attack",
    "shelling", "bombardment", "blast", "black sea fleet", "blackout",
    "power outage",
]

RAW_DIR  = NEWS_DIR / "data" / "raw"
PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in ANALYSIS_KEYWORDS)


for raw_file in sorted(RAW_DIR.glob("warzone*.json")):
    try:
        data = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to load {raw_file}: {exc}")
        continue

    if isinstance(data, dict) and data.get("articles"):
        records = data.get("articles")
    elif isinstance(data, list):
        records = data
    else:
        records = []

    rows = []
    for record in records:
        title = record.get("title", "")
        link = record.get("url", "")
        if not title or not link:
            continue

        if not matches_keywords(f"{title} {record.get('category', '')} {record.get('author', '')}"):
            continue

        rows.append(
            {
                "date": parse_datetime(record.get("date", "")),
                "news source": "War Zone",
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching records found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed WarZone to: {processed_path} ({len(df)} rows)")


**What this notebook does**
- Scrapes The War Zone search results for the selected keywords and saves the raw JSON under `data/raw/`.
- Re-processes each saved raw file into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read the scraped article records from the raw JSON.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse the scraped date into a full datetime and default the time to 12:00 when only a date is available.
- Set the news source label to `War Zone`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant articles.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
` Merging all news`


In [ ]:
# Incremental merge for War Zone: keep history in warzone_all.csv and append newcomers
import pandas as pd

PROC_DIR = NEWS_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "warzone_all.csv"

wz_files = sorted(
    f for f in PROC_DIR.glob("warzone*.csv")
    if f.name.lower() != "warzone_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in wz_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No War Zone data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental War Zone master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None


--------
# Task 2: Spatial & Temporal Analysis

### Data Cleaning and Preparation

This notebook cleans FIRMS CSV files from the raw folder and writes the cleaned versions to the processed folder.

Cleaning rules:
- drop all rows with missing values
- for MODIS rows, keep only `confidence > 75` and `frp > 20`
- for VIIRS rows, drop rows where confidence is `l` and keep only `frp > 5`

The code is written to handle future raw FIRMS CSVs too, not only the current Ukraine and Turkey files. Cleaned files use the same name pattern as the raw file, but with `all` replaced by `clnd`.

` The code below sets up the function `

In [2]:
from pathlib import Path
import re

import geopandas as gpd
import pandas as pd
from IPython.display import display

# Use pre-resolved paths from Cell 3 (ROOT, FIRMS_DIR, GADM_DIR)
RAW_DIR       = FIRMS_DIR / "data" / "raw"
PROCESSED_DIR = FIRMS_DIR / "data" / "processed"
BOUNDARIES_DIR = GADM_DIR
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:       ", RAW_DIR)
print("Processed folder: ", PROCESSED_DIR)
print("Boundaries folder:", BOUNDARIES_DIR)

COMMON_REQUIRED_COLUMNS = [
    "latitude", "longitude", "scan", "track", "acq_date", "acq_time",
    "satellite", "instrument", "confidence", "version", "frp", "daynight", "type",
]

MODIS_REQUIRED_COLUMNS = COMMON_REQUIRED_COLUMNS + ["brightness", "bright_t31"]
VIIRS_REQUIRED_COLUMNS = COMMON_REQUIRED_COLUMNS + ["bright_ti4", "bright_ti5"]

NORMALIZED_OUTPUT_COLUMNS = [
    "latitude", "longitude", "brightness", "scan", "track", "acq_date",
    "acq_time", "satellite", "confidence", "bright_t31", "frp", "daynight",
    "country", "area_label", "bright_ti4",
]

# --- GADM boundary map: country slug -> .gpkg file (all available) ---
COUNTRY_BOUNDARY_FILES = {
    "ukraine":      BOUNDARIES_DIR / "gadm41_UKR.gpkg",
    "turkey":       BOUNDARIES_DIR / "gadm41_TUR.gpkg",
    "iran":         BOUNDARIES_DIR / "gadm41_IRN.gpkg",
    "israel":       BOUNDARIES_DIR / "gadm41_ISR.gpkg",
    "qatar":        BOUNDARIES_DIR / "gadm41_QAT.gpkg",
    "uae":          BOUNDARIES_DIR / "gadm41_ARE.gpkg",
    "australia":    BOUNDARIES_DIR / "gadm41_AUS.gpkg",
}

# Filter to only files that exist on disk
COUNTRY_BOUNDARY_FILES = {k: v for k, v in COUNTRY_BOUNDARY_FILES.items() if v.exists()}
print("Available GADM boundaries:", list(COUNTRY_BOUNDARY_FILES.keys()))


def infer_row_family(frame):
    """Classify each row as MODIS, VIIRS, or UNKNOWN using satellite/instrument text."""
    source_text = (
        frame.get("satellite", pd.Series(index=frame.index, dtype="string")).astype(str).str.lower()
        + " "
        + frame.get("instrument", pd.Series(index=frame.index, dtype="string")).astype(str).str.lower()
    )
    family = pd.Series("unknown", index=frame.index, dtype="string")
    family[source_text.str.contains("modis", na=False)] = "modis"
    family[source_text.str.contains("viirs", na=False)] = "viirs"
    return family


def cleaned_name(raw_path):
    """Turn all_*.csv into clnd_*.csv and keep the rest of the filename stable."""
    stem = raw_path.stem
    if stem.startswith("all_"):
        stem = "clnd_" + stem[len("all_"):]
    elif stem == "all":
        stem = "clnd"
    else:
        stem = "clnd_" + stem
    return raw_path.with_name(stem + raw_path.suffix)


def required_columns_for_family(family_name):
    if family_name == "modis":
        return MODIS_REQUIRED_COLUMNS
    if family_name == "viirs":
        return VIIRS_REQUIRED_COLUMNS
    return COMMON_REQUIRED_COLUMNS


def boundary_file_for_dataset(dataset_name):
    lower_name = dataset_name.lower()
    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
        if country_name in lower_name:
            return country_name, boundary_path
    return None, None


def apply_gadm_cleaning(frame, dataset_name):
    """Keep only points inside country boundary and label area with GADM Admin-1 region names."""
    cleaned = frame.copy()
    if cleaned.empty:
        return cleaned

    cleaned["latitude"] = pd.to_numeric(cleaned.get("latitude"), errors="coerce")
    cleaned["longitude"] = pd.to_numeric(cleaned.get("longitude"), errors="coerce")
    cleaned = cleaned.dropna(subset=["latitude", "longitude"]).copy()
    if cleaned.empty:
        return cleaned

    country_name, boundary_path = boundary_file_for_dataset(dataset_name)
    cleaned["country"] = country_name.title() if country_name else None
    if boundary_path is None or not boundary_path.exists():
        return cleaned

    points_gdf = gpd.GeoDataFrame(
        cleaned,
        geometry=gpd.points_from_xy(cleaned["longitude"], cleaned["latitude"]),
        crs="EPSG:4326",
    )

    country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
    country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)

    points_gdf = gpd.clip(points_gdf, country_admin0[["geometry"]])
    if points_gdf.empty:
        return cleaned.iloc[0:0].copy()

    region_name_column = "NAME_1"
    if region_name_column not in country_admin1.columns:
        fallback_names = [column for column in country_admin1.columns if column.startswith("NAME")]
        if fallback_names:
            region_name_column = fallback_names[0]
        else:
            region_name_column = None

    if region_name_column is not None:
        admin1_labels = country_admin1[[region_name_column, "geometry"]].rename(
            columns={region_name_column: "gadm_region"}
        )
        points_gdf = gpd.sjoin(points_gdf, admin1_labels, how="left", predicate="intersects")

        existing_area_label = (
            points_gdf["area_label"] if "area_label" in points_gdf.columns else pd.Series(index=points_gdf.index, dtype="string")
        )
        points_gdf["area_label"] = points_gdf["gadm_region"].fillna(existing_area_label)
        points_gdf["area_label"] = points_gdf["area_label"].fillna(country_name.title())

        drop_columns = [column for column in ["gadm_region", "index_right"] if column in points_gdf.columns]
        points_gdf = points_gdf.drop(columns=drop_columns)

    points_gdf = points_gdf.drop(columns=["geometry"])
    return pd.DataFrame(points_gdf)


def clean_firms_frame(frame):
    """Drop missing rows using sensor-specific required fields and apply MODIS/VIIRS quality rules."""
    working = frame.copy()
    family = infer_row_family(working)
    cleaned_frames = []

    for family_name in ["modis", "viirs", "unknown"]:
        family_mask = family.eq(family_name)
        if not family_mask.any():
            continue

        family_rows = working.loc[family_mask].copy()
        required_columns = [column for column in required_columns_for_family(family_name) if column in family_rows.columns]
        family_rows = family_rows.dropna(subset=required_columns, how="any")

        if family_name == "modis":
            modis_conf = pd.to_numeric(family_rows["confidence"], errors="coerce")
            modis_frp = pd.to_numeric(family_rows["frp"], errors="coerce")
            family_rows = family_rows.loc[(modis_conf > 75) & (modis_frp > 20)]
        elif family_name == "viirs":
            viirs_conf = family_rows["confidence"].astype(str).str.strip().str.lower()
            viirs_frp = pd.to_numeric(family_rows["frp"], errors="coerce")
            family_rows = family_rows.loc[(viirs_conf != "l") & (viirs_frp > 5)]
        else:
            family_conf = pd.to_numeric(family_rows["confidence"], errors="coerce")
            family_frp = pd.to_numeric(family_rows["frp"], errors="coerce")
            family_rows = family_rows.loc[family_conf.notna() & family_frp.notna()]

        family_rows["cleaning_family"] = family_name
        cleaned_frames.append(family_rows)

    if not cleaned_frames:
        return working.iloc[0:0].copy()

    cleaned = pd.concat(cleaned_frames, ignore_index=True)
    return cleaned


def clean_raw_file(raw_path):
    """Clean one raw FIRMS CSV and write the result to the processed folder."""
    frame = pd.read_csv(raw_path)
    cleaned = clean_firms_frame(frame)
    cleaned = apply_gadm_cleaning(cleaned, raw_path.stem)
    available_output_columns = [column for column in NORMALIZED_OUTPUT_COLUMNS if column in cleaned.columns]
    cleaned = cleaned.loc[:, available_output_columns].copy()

    output_path = PROCESSED_DIR / cleaned_name(raw_path).name
    cleaned.to_csv(output_path, index=False)
    return output_path, len(frame), len(cleaned)


Raw folder:        C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\raw
Processed folder:  C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\processed
Boundaries folder: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\Boundaries\GADM boundaries
Available GADM boundaries: ['ukraine', 'turkey', 'iran', 'israel', 'qatar', 'uae', 'australia']


` Execute the functions for data in 'raw' folder `

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv"))

if not raw_files:
    print("No CSV files found in:", RAW_DIR)
else:
    results = []
    for raw_file in raw_files:
        output_path, before_rows, after_rows = clean_raw_file(raw_file)
        results.append({
            "raw_file": raw_file.name,
            "processed_file": output_path.name,
            "rows_before": before_rows,
            "rows_after": after_rows,
        })
        print(f"{raw_file.name} -> {output_path.name} | {before_rows} rows -> {after_rows} rows")

    summary = pd.DataFrame(results)
    print("\nCleaning summary:")
    display(summary)

C:\Users\AFT\AppData\Local\Temp\ipykernel_31204\1611580302.py:186: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  frame = pd.read_csv(raw_path)


---
### Mapping after cleaning

In [ ]:
# Generate an interactive map for each cleaned (`clnd_*.csv`) dataset using the matching GADM boundaries.
#from pathlib import Path
#
#import geopandas as gpd
#import pandas as pd
#import folium
#from folium.plugins import HeatMap
#
#OUTPUT_DIR = BASE_DIR / "outputs" / "maps"
#BOUNDARIES_DIR = BASE_DIR.parent / "boundaries" / "GADM boundaries"
#OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
#COUNTRY_BOUNDARY_FILES = {
#    "ukraine": BOUNDARIES_DIR / "gadm41_UKR.gpkg",
#    "turkey": BOUNDARIES_DIR / "gadm41_TUR.gpkg",
#}
#
#
#def boundary_file_for_dataset(dataset_name):
#    lower_name = dataset_name.lower()
#    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
#        if country_name in lower_name:
#            return country_name, boundary_path
#    return None, None
#
#
#processed_files = sorted(PROCESSED_DIR.glob("clnd_*.csv"))
#if not processed_files:
#    print("No cleaned files found in:", PROCESSED_DIR)
#else:
#    for pf in processed_files:
#        country_name, boundary_path = boundary_file_for_dataset(pf.stem)
#        if boundary_path is None or not boundary_path.exists():
#            print(f"{pf.name}: no matching GADM boundary file found, skipping")
#            continue
#
#        print("Processing:", pf.name)
#        print("Using boundary:", boundary_path.name)
#
#        df = pd.read_csv(pf)
#        df["latitude"] = pd.to_numeric(df.get("latitude"), errors="coerce")
#        df["longitude"] = pd.to_numeric(df.get("longitude"), errors="coerce")
#        df = df.dropna(subset=["latitude", "longitude"]).copy()
#        if df.empty:
#            print(f"{pf.name}: no valid points, skipping")
#            continue
#
#        points_gdf = gpd.GeoDataFrame(
#            df,
#            geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
#            crs="EPSG:4326",
#        )
#
#        country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
#        country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)
#
#        country_points = gpd.clip(points_gdf, country_admin0[["geometry"]])
#        country_points = country_points.dropna(subset=["geometry"]).copy()
#        if country_points.empty:
#            print(f"{pf.name}: no points inside {country_name}, skipping")
#            continue
#
#        center_lat = country_points.geometry.y.median()
#        center_lon = country_points.geometry.x.median()
#        m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")
#
#        folium.GeoJson(
#            country_admin0.to_json(),
#            name=f"{country_name.title()} boundary",
#            style_function=lambda _: {"color": "#111827", "weight": 2.2, "fillOpacity": 0},
#        ).add_to(m)
#
#        folium.GeoJson(
#            country_admin1.to_json(),
#            name="Admin-1 boundaries",
#            style_function=lambda _: {"color": "#6b7280", "weight": 0.8, "fillOpacity": 0},
#        ).add_to(m)
#
#        heat_data = []
#        for _, row in country_points.iterrows():
#            w = row.get("brightness")
#            if pd.isna(w):
#                w = row.get("frp")
#            try:
#                weight = float(w) if pd.notna(w) else 1.0
#            except Exception:
#                weight = 1.0
#            heat_data.append([row["latitude"], row["longitude"], weight])
#
#        if heat_data:
#            HeatMap(heat_data, radius=12, blur=10, min_opacity=0.3, name="Fire intensity").add_to(m)
#
#        for _, row in country_points.iterrows():
#            daynight = row.get("daynight", "")
#            color = "#ef4444" if daynight == "D" else "#f59e0b"
#
#            # Marker size is scaled using FRP only.
#            frp_value = row.get("frp")
#            try:
#                frp_value = float(frp_value) if pd.notna(frp_value) else 0.0
#            except Exception:
#                frp_value = 0.0
#            radius = 3 + min(frp_value / 10.0, 9)
#
#            popup_text = (
#                f"Satellite: {row.get('satellite', '')}<br>"
#                f"Date: {row.get('acq_date', '')} {row.get('acq_time', '')}<br>"
#                f"FRP: {row.get('frp', '')}<br>"
#                f"Confidence: {row.get('confidence', '')}<br>"
#                f"Area: {row.get('area_label', '')}"
#            )
#            folium.CircleMarker(
#                location=[row["latitude"], row["longitude"]],
#                radius=radius,
#                color=color,
#                weight=1,
#                fill=True,
#                fill_color=color,
#                fill_opacity=0.75,
#                popup=folium.Popup(popup_text, max_width=300),
#            ).add_to(m)
#
#        folium.LayerControl(collapsed=False).add_to(m)
#        outpath = OUTPUT_DIR / (pf.stem + ".html")
#        m.save(str(outpath))
#        print(f"Saved map: {outpath}")

---
### Clustering

In [ ]:
# Cluster cleaned FIRMS points using time and distance thresholds, then summarize each cluster.
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
from IPython.display import display

# Use pre-resolved paths from Cell 3 (ROOT, FIRMS_DIR, GADM_DIR)
PROCESSED_DIR = FIRMS_DIR / "data" / "processed"
BOUNDARIES_DIR = GADM_DIR

COUNTRY_BOUNDARY_FILES = {
    "ukraine":      BOUNDARIES_DIR / "gadm41_UKR.gpkg",
    "turkey":       BOUNDARIES_DIR / "gadm41_TUR.gpkg",
    "iran":         BOUNDARIES_DIR / "gadm41_IRN.gpkg",
    "israel":       BOUNDARIES_DIR / "gadm41_ISR.gpkg",
    "qatar":        BOUNDARIES_DIR / "gadm41_QAT.gpkg",
    "uae":          BOUNDARIES_DIR / "gadm41_ARE.gpkg",
    "australia":    BOUNDARIES_DIR / "gadm41_AUS.gpkg",
}
# Filter to only files that exist on disk
COUNTRY_BOUNDARY_FILES = {k: v for k, v in COUNTRY_BOUNDARY_FILES.items() if v.exists()}


# Use pre-resolved paths from Cell 3 (ROOT, FIRMS_DIR, GADM_DIR)
PROCESSED_DIR = FIRMS_DIR / "data" / "processed"
BOUNDARIES_DIR = GADM_DIR

COUNTRY_BOUNDARY_FILES = {
    "ukraine":      BOUNDARIES_DIR / "gadm41_UKR.gpkg",
    "turkey":       BOUNDARIES_DIR / "gadm41_TUR.gpkg",
    "iran":         BOUNDARIES_DIR / "gadm41_IRN.gpkg",
    "israel":       BOUNDARIES_DIR / "gadm41_ISR.gpkg",
    "qatar":        BOUNDARIES_DIR / "gadm41_QAT.gpkg",
    "uae":          BOUNDARIES_DIR / "gadm41_ARE.gpkg",
    "australia":    BOUNDARIES_DIR / "gadm41_AUS.gpkg",
}
# Filter to only files that exist on disk
COUNTRY_BOUNDARY_FILES = {k: v for k, v in COUNTRY_BOUNDARY_FILES.items() if v.exists()}


# Use pre-resolved paths from Cell 3 (ROOT, FIRMS_DIR, GADM_DIR)
PROCESSED_DIR = FIRMS_DIR / "data" / "processed"
BOUNDARIES_DIR = GADM_DIR

COUNTRY_BOUNDARY_FILES = {
    "ukraine":      BOUNDARIES_DIR / "gadm41_UKR.gpkg",
    "turkey":       BOUNDARIES_DIR / "gadm41_TUR.gpkg",
    "iran":         BOUNDARIES_DIR / "gadm41_IRN.gpkg",
    "israel":       BOUNDARIES_DIR / "gadm41_ISR.gpkg",
    "qatar":        BOUNDARIES_DIR / "gadm41_QAT.gpkg",
    "uae":          BOUNDARIES_DIR / "gadm41_ARE.gpkg",
    "australia":    BOUNDARIES_DIR / "gadm41_AUS.gpkg",
}
# Filter to only files that exist on disk
COUNTRY_BOUNDARY_FILES = {k: v for k, v in COUNTRY_BOUNDARY_FILES.items() if v.exists()}


TIME_WINDOW_HOURS = 12
DISTANCE_THRESHOLDS_M = {
    "modis": 2000.0,
    "viirs": 800.0,
}

CLUSTER_OUTPUT_COLUMNS = [
    "sensor",
    "n_points",
    "centroid_lat",
    "centroid_lon",
    "max_brightness",
    "maxbrightness_ti4",
    "average_confidence_rate",
    "sum_frp",
    "max_frp",
    "earliest_detection",
    "latest_detection",
    "duration_hours",
    "centroid_country",
    "centroid_area_label",
]


def infer_sensor_family_from_satellite(series):
    sat = series.astype(str).str.lower()
    family = pd.Series("unknown", index=series.index, dtype="string")
    family[sat.str.contains("modis", na=False)] = "modis"
    family[sat.str.contains("viirs", na=False)] = "viirs"
    return family


def parse_detection_datetime(frame):
    date_part = pd.to_datetime(frame.get("acq_date"), errors="coerce")
    time_raw = frame.get("acq_time", pd.Series(index=frame.index, dtype="object")).astype(str)
    time_digits = time_raw.str.extract(r"(\d+)", expand=False).fillna("0").str.zfill(4).str[-4:]

    hours = pd.to_numeric(time_digits.str[:2], errors="coerce").fillna(0).clip(lower=0, upper=23)
    minutes = pd.to_numeric(time_digits.str[2:], errors="coerce").fillna(0).clip(lower=0, upper=59)
    return date_part + pd.to_timedelta(hours * 60 + minutes, unit="m")


def boundary_file_for_dataset_name(dataset_name):
    lower_name = dataset_name.lower()
    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
        if country_name in lower_name:
            return country_name, boundary_path
    return None, None


def clustered_name_from_cleaned(cleaned_name):
    stem = Path(cleaned_name).stem
    if stem.startswith("clnd_"):
        return "clstrd_" + stem[len("clnd_"):] + ".csv"
    if stem == "clnd":
        return "clstrd.csv"
    return "clstrd_" + stem + ".csv"


def union_find_labels(geometries, times, eps_m, max_hours):
    n = len(geometries)
    if n == 0:
        return np.array([], dtype=int)

    parent = np.arange(n)
    rank = np.zeros(n, dtype=int)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    sindex = geometries.sindex

    for i, geom_i in enumerate(geometries):
        bounds = geom_i.buffer(eps_m).bounds
        candidates = list(sindex.intersection(bounds))
        time_i = times[i]
        for j in candidates:
            if j <= i:
                continue
            delta_hours = abs((times[j] - time_i).total_seconds()) / 3600.0
            if delta_hours > max_hours:
                continue
            if geom_i.distance(geometries.iloc[j]) <= eps_m:
                union(i, j)

    roots = np.array([find(i) for i in range(n)])
    root_to_label = {}
    labels = np.zeros(n, dtype=int)
    next_label = 0
    for idx, root in enumerate(roots):
        if root not in root_to_label:
            root_to_label[root] = next_label
            next_label += 1
        labels[idx] = root_to_label[root]
    return labels


def keep_consecutive_time_runs(frame, label_col, time_col, max_hours):
    """Keep only runs where consecutive detections are <= max_hours; singleton runs are dropped."""
    kept_runs = []
    next_cluster_id = 0

    for _, grp in frame.groupby(label_col, sort=False):
        grp = grp.sort_values(time_col).copy()
        idx_list = grp.index.tolist()
        if not idx_list:
            continue

        run = [idx_list[0]]
        for curr_idx in idx_list[1:]:
            prev_idx = run[-1]
            gap_h = (grp.loc[curr_idx, time_col] - grp.loc[prev_idx, time_col]).total_seconds() / 3600.0
            if gap_h <= max_hours:
                run.append(curr_idx)
            else:
                if len(run) >= 2:
                    run_df = grp.loc[run].copy()
                    run_df[label_col] = next_cluster_id
                    kept_runs.append(run_df)
                    next_cluster_id += 1
                run = [curr_idx]

        if len(run) >= 2:
            run_df = grp.loc[run].copy()
            run_df[label_col] = next_cluster_id
            kept_runs.append(run_df)
            next_cluster_id += 1

    if not kept_runs:
        return frame.iloc[0:0].copy()
    return pd.concat(kept_runs, ignore_index=True)


def assign_centroid_area(cluster_frame, country_admin1, country_name):
    centroids_gdf = gpd.GeoDataFrame(
        cluster_frame.copy(),
        geometry=gpd.points_from_xy(cluster_frame["centroid_lon"], cluster_frame["centroid_lat"]),
        crs="EPSG:4326",
    )
    cluster_frame["centroid_country"] = country_name.title()

    region_name_column = "NAME_1"
    if region_name_column not in country_admin1.columns:
        name_candidates = [column for column in country_admin1.columns if column.startswith("NAME")]
        region_name_column = name_candidates[0] if name_candidates else None

    if region_name_column is None:
        cluster_frame["centroid_area_label"] = country_name.title()
        return cluster_frame

    labels = country_admin1[[region_name_column, "geometry"]].rename(columns={region_name_column: "centroid_area_label"})
    joined = gpd.sjoin(centroids_gdf, labels, how="left", predicate="intersects")
    cluster_frame["centroid_area_label"] = joined["centroid_area_label"].fillna(country_name.title())
    return cluster_frame


processed_files = sorted(PROCESSED_DIR.glob("clnd_*.csv"))
if not processed_files:
    print("No cleaned files found in:", PROCESSED_DIR)
else:
    for pf in processed_files:
        dataset_name = pf.stem
        print("\nClustering:", pf.name)

        country_name, boundary_path = boundary_file_for_dataset_name(dataset_name)
        if boundary_path is None or not boundary_path.exists():
            print(f"{pf.name}: no matching GADM boundary file found, skipping")
            continue

        df = pd.read_csv(pf)
        if df.empty:
            print(f"{pf.name}: empty dataset, skipping")
            continue

        df["latitude"] = pd.to_numeric(df.get("latitude"), errors="coerce")
        df["longitude"] = pd.to_numeric(df.get("longitude"), errors="coerce")
        df["brightness"] = pd.to_numeric(df.get("brightness"), errors="coerce")
        df["frp"] = pd.to_numeric(df.get("frp"), errors="coerce")
        df["detected_at"] = parse_detection_datetime(df)
        df = df.dropna(subset=["latitude", "longitude", "detected_at"]).copy()
        if df.empty:
            print(f"{pf.name}: no valid rows after datetime/coordinate parsing")
            continue

        # Ensure only in-boundary points are clustered.
        points_gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
            crs="EPSG:4326",
        )
        country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
        country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)
        points_gdf = gpd.clip(points_gdf, country_admin0[["geometry"]])
        points_gdf = points_gdf.dropna(subset=["geometry"]).copy()
        if points_gdf.empty:
            print(f"{pf.name}: no points remain inside boundary")
            continue

        points_gdf["sensor_family"] = infer_sensor_family_from_satellite(points_gdf["satellite"])
        dataset_clusters = []

        for family_name, eps_m in DISTANCE_THRESHOLDS_M.items():
            family_points = points_gdf.loc[points_gdf["sensor_family"] == family_name].copy()
            if family_points.empty:
                continue

            projected_crs = family_points.estimate_utm_crs()
            family_proj = family_points.to_crs(projected_crs)
            family_proj = family_proj.sort_values("detected_at").reset_index(drop=True)

            labels = union_find_labels(
                family_proj.geometry,
                family_proj["detected_at"].tolist(),
                eps_m=eps_m,
                max_hours=TIME_WINDOW_HOURS,
            )

            family_proj["cluster_local_id"] = labels
            family_geo = family_proj.to_crs(epsg=4326)

            # Keep only consecutive (<=6h) runs and drop points with >6h gaps from runs.
            family_geo = keep_consecutive_time_runs(
                family_geo,
                label_col="cluster_local_id",
                time_col="detected_at",
                max_hours=TIME_WINDOW_HOURS,
            )
            if family_geo.empty:
                continue

            # Aggregate per cluster.
            for cluster_id, grp in family_geo.groupby("cluster_local_id", sort=True):
                earliest = grp["detected_at"].min()
                latest = grp["detected_at"].max()
                duration_h = (latest - earliest).total_seconds() / 3600.0
                cluster_geom = gpd.GeoSeries(grp.geometry, crs="EPSG:4326").union_all().centroid

                # Compute average confidence rate
                if family_name == "modis":
                    avg_conf = grp["confidence"].dropna().astype(float).mean()
                    avg_conf = round(avg_conf, 1) if pd.notna(avg_conf) else np.nan
                else:
                    conf_series = grp["confidence"].dropna().astype(str).str.strip().str.lower()
                    avg_conf = conf_series.mode()[0] if not conf_series.empty else "n"

                # Compute maxbrightness_ti4
                if "bright_ti4" in grp.columns and grp["bright_ti4"].notna().any():
                    max_ti4 = float(grp["bright_ti4"].max())
                else:
                    max_ti4 = np.nan

                dataset_clusters.append(
                    {
                        "sensor": family_name,
                        "n_points": int(len(grp)),
                        "centroid_lat": float(cluster_geom.y),
                        "centroid_lon": float(cluster_geom.x),
                        "max_brightness": float(grp["brightness"].max()) if grp["brightness"].notna().any() else np.nan,
                        "maxbrightness_ti4": max_ti4,
                        "average_confidence_rate": avg_conf,
                        "sum_frp": float(grp["frp"].sum(skipna=True)),
                        "max_frp": float(grp["frp"].max()) if grp["frp"].notna().any() else np.nan,
                        "earliest_detection": earliest,
                        "latest_detection": latest,
                        "duration_hours": duration_h,
                    }
                )

        if not dataset_clusters:
            print(f"{pf.name}: no clusters produced")
            continue

        cluster_table = pd.DataFrame(dataset_clusters)
        cluster_table = assign_centroid_area(cluster_table, country_admin1, country_name)
        cluster_table = cluster_table.loc[:, CLUSTER_OUTPUT_COLUMNS].copy()
        cluster_table = cluster_table.sort_values(["sensor", "earliest_detection", "n_points"], ascending=[True, True, False])

        out_file = PROCESSED_DIR / clustered_name_from_cleaned(pf.name)
        cluster_table.to_csv(out_file, index=False)
        print(f"Saved cluster file: {out_file} | clusters: {len(cluster_table)}")
        display(cluster_table.head(10))

`Clustered Map`

In [ ]:
# Generate an interactive map for each clustered (`clstrd_*.csv`) dataset using matching GADM boundaries.
#import folium
#import pandas as pd
#from pathlib import Path
#import geopandas as gpd
#
#CLUSTER_MAP_DIR = BASE_DIR / "outputs" / "maps"
#CLUSTER_MAP_DIR.mkdir(parents=True, exist_ok=True)
#
#
#def boundary_file_for_clustered_dataset(dataset_name):
#    lower_name = dataset_name.lower()
#    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
#        if country_name in lower_name:
#            return country_name, boundary_path
#    return None, None
#
#
#clustered_files = sorted(PROCESSED_DIR.glob("clstrd_*.csv"))
#if not clustered_files:
#    print("No clustered files found in:", PROCESSED_DIR)
#else:
#    for cf in clustered_files:
#        dataset_name = cf.stem
#        country_name, boundary_path = boundary_file_for_clustered_dataset(dataset_name)
#        if boundary_path is None or not boundary_path.exists():
#            print(f"{cf.name}: no matching GADM boundary file found, skipping")
#            continue
#
#        print("Processing clustered file:", cf.name)
#        print("Using boundary:", boundary_path.name)
#
#        cluster_df = pd.read_csv(cf)
#        cluster_df["centroid_lat"] = pd.to_numeric(cluster_df.get("centroid_lat"), errors="coerce")
#        cluster_df["centroid_lon"] = pd.to_numeric(cluster_df.get("centroid_lon"), errors="coerce")
#        cluster_df["sum_frp"] = pd.to_numeric(cluster_df.get("sum_frp"), errors="coerce")
#        cluster_df["max_frp"] = pd.to_numeric(cluster_df.get("max_frp"), errors="coerce")
#        cluster_df = cluster_df.dropna(subset=["centroid_lat", "centroid_lon"]).copy()
#        if cluster_df.empty:
#            print(f"{cf.name}: no valid centroid rows, skipping")
#            continue
#
#        country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
#        country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)
#
#        center_lat = cluster_df["centroid_lat"].median()
#        center_lon = cluster_df["centroid_lon"].median()
#        m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")
#
#        folium.GeoJson(
#            country_admin0.to_json(),
#            name=f"{country_name.title()} boundary",
#            style_function=lambda _: {"color": "#111827", "weight": 2.2, "fillOpacity": 0},
#        ).add_to(m)
#
#        folium.GeoJson(
#            country_admin1.to_json(),
#            name="Admin-1 boundaries",
#            style_function=lambda _: {"color": "#6b7280", "weight": 0.8, "fillOpacity": 0},
#        ).add_to(m)
#
#        for _, row in cluster_df.iterrows():
#            sensor = str(row.get("sensor", "")).lower()
#            color = "#2563eb" if sensor == "modis" else "#ef4444"
#
#            sum_frp_val = row.get("sum_frp")
#            if pd.isna(sum_frp_val):
#                sum_frp_val = row.get("max_frp")
#            try:
#                sum_frp_val = float(sum_frp_val) if pd.notna(sum_frp_val) else 0.0
#            except Exception:
#                sum_frp_val = 0.0
#
#            radius = 4 + min(sum_frp_val / 20.0, 10)
#
#            popup_text = (
#                f"Sensor: {row.get('sensor', '')}<br>"
#                f"Points: {row.get('n_points', '')}<br>"
#                f"Max Brightness: {row.get('max_brightness', '')}<br>"
#                f"Maxbrightness TI4: {row.get('maxbrightness_ti4', '')}<br>"
#                f"Average Confidence: {row.get('average_confidence_rate', '')}<br>"
#                f"Sum FRP: {row.get('sum_frp', '')}<br>"
#                f"Max FRP: {row.get('max_frp', '')}<br>"
#                f"Earliest: {row.get('earliest_detection', '')}<br>"
#                f"Latest: {row.get('latest_detection', '')}<br>"
#                f"Duration (h): {row.get('duration_hours', '')}<br>"
#                f"Centroid Country: {row.get('centroid_country', '')}<br>"
#                f"Centroid Area: {row.get('centroid_area_label', '')}"
#            )
#
#            folium.CircleMarker(
#                location=[row["centroid_lat"], row["centroid_lon"]],
#                radius=radius,
#                color=color,
#                weight=1,
#                fill=True,
#                fill_color=color,
#                fill_opacity=0.75,
#                popup=folium.Popup(popup_text, max_width=320),
#            ).add_to(m)
#
#        folium.LayerControl(collapsed=False).add_to(m)
#
#        html_name = dataset_name + "_map.html"
#        outpath = CLUSTER_MAP_DIR / html_name
#        m.save(str(outpath))
#        print(f"Saved clustered map: {outpath}")

---
# Task 3: Thermal-News Correlation & Classification


## Correlation Code


In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

# Chapter 6: correlate clustered thermal events with processed news tables.
#
# Workflow:
# 1. Load all FIRMS clustered event files from FIRMS/data/processed/clstrd_*.csv.
# 2. Load all combined news tables from News/data/processed/*_all.csv.
# 3. For each thermal event, find news inside a 24 hour window.
# 4. Pick the nearest news item first, then score it by content and area overlap.
# 5. Keep every thermal event in the output, even if no news match exists.
# 6. Save the results back to FIRMS/data/processed/ as crrltd_*.csv.

# ROOT, FIRMS_DIR, NEWS_DIR are set in Cell 3
FIRMS_PROCESSED = FIRMS_DIR / "data" / "processed"
NEWS_PROCESSED  = NEWS_DIR  / "data" / "processed"
MAX_TIME_WINDOW_HOURS = 24
MIN_ACTION_HITS = 1
MIN_CONFLICT_HITS = 3

ACTION_IMPACT_TERMS = [
    "attack",
    "attacked",
    "attacks",
    "strike",
    "struck",
    "strikes",
    "striked",
    "explosion",
    "exploded",
    "explodes",
    "explode",
    "blast",
    "blasted",
    "bomb",
    "bombed",
    "bombing",
    "killed",
    "killing",
    "injured",
    "injury",
    "hit",
    "hits",
    "shelling",
    "bombardment",
    "burning",
    "burned",
    "fire",
]

CONFLICT_TERMS = [
    "war",
    "missile",
    "missiles",
    "rocket",
    "rockets",
    "drone",
    "drones",
    "artillery",
    "invasion",
    "airstrike",
    "airstrikes",
    "raid",
    "raids",
    "frontline",
    "combat",
    "conflict",
    "ukraine",
    "ukrainian",
    "russia",
    "russian",
    "kyiv",
    "kiev",
    "kharkiv",
    "odesa",
    "odessa",
    "crimea",
    "donbas",
    "black sea",
    "zaporizhzhia",
    "energy grid",
    "critical infrastructure",
]

FIRE_TERMS = [
    "wildfire",
    "forest fire",
    "forestfire",
    "fire",
    "brush fire",
    "bushfire",
    "blaze",
    "inferno",
    "arson",
    "flames",
    "burning",
    "smoke",
    "spread",
    "spreaded",
    "wind",
    "burn",
    "burned",
]

pd.set_option("display.max_colwidth", 120)

EXTRA_COLUMNS = [
    "matched_news_count",
    "nearest_delta_time_hours",
    "time_score",
    "action_impact_score",
    "conflict_word_score",
    "fire_word_score",
    "area_score",
    "keyword_hit_count",
    "high_confidence_match",
    "total_score",
    "nearest_news_source",
    "nearest_news_title",
]


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = text.replace("'", " ")
    text = re.sub(r"[^a-z0-9\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def unique_hits(text, terms):
    normalized = normalize_text(text)
    hits = []
    for term in terms:
        normalized_term = normalize_text(term)
        if not normalized_term:
            continue
        pattern = rf"\b{re.escape(normalized_term)}\b"
        if re.search(pattern, normalized):
            hits.append(term)
    return sorted(set(hits))


def load_news_corpus(news_dir):
    frames = []
    for news_path in sorted(news_dir.glob("*_all.csv")):
        frame = pd.read_csv(news_path)
        frame = frame.copy()
        frame["news_file"] = news_path.name
        frame["date"] = pd.to_datetime(frame["date"], errors="coerce", utc=True)
        frame = frame.dropna(subset=["date", "title"])
        frames.append(frame)

    if not frames:
        raise FileNotFoundError(f"No *_all.csv files were found in {news_dir}")

    news = pd.concat(frames, ignore_index=True)
    return news.sort_values("date").reset_index(drop=True)


def load_firms_events(cluster_path):
    events = pd.read_csv(cluster_path)
    events = events.copy()
    events["source_file"] = cluster_path.name
    events["earliest_detection"] = pd.to_datetime(events["earliest_detection"], errors="coerce", utc=True)
    events["latest_detection"] = pd.to_datetime(events["latest_detection"], errors="coerce", utc=True)
    events = events.dropna(subset=["earliest_detection"])
    return events


def reference_time_for_event(row):
    start = row["earliest_detection"]
    end = row["latest_detection"]
    if pd.isna(end):
        return start
    return start + (end - start) / 2


def score_candidate(reference_time, news_row, country_name, area_name):
    news_text = f"{news_row.get('title', '')} {news_row.get('news source', '')}".lower()
    
    country_lower = str(country_name).lower() if pd.notna(country_name) else ""
    area_lower = str(area_name).lower() if pd.notna(area_name) else ""
    
    country_search_terms = []
    if country_lower:
        country_search_terms.append(country_lower)
        if country_lower == "turkey":
            country_search_terms.append("turkiye")
        elif country_lower == "ukraine":
            country_search_terms.append("ukrainian")
            
    country_present = any(len(unique_hits(news_text, [term])) > 0 for term in country_search_terms) if country_search_terms else False
    
    area_search_terms = [area_lower] if area_lower else []
    area_present = any(len(unique_hits(news_text, [term])) > 0 for term in area_search_terms) if area_search_terms else False
    
    if not (country_present or area_present):
        return None

    action_hits = unique_hits(news_text, ACTION_IMPACT_TERMS)
    conflict_hits = unique_hits(news_text, CONFLICT_TERMS)
    fire_hits = unique_hits(news_text, FIRE_TERMS)

    delta_hours = abs((news_row["date"] - reference_time).total_seconds()) / 3600.0
    if delta_hours > MAX_TIME_WINDOW_HOURS:
        return None

    action_count = len(action_hits)
    conflict_count = len(conflict_hits)
    fire_count = len(fire_hits)

    time_score = max(0.0, (MAX_TIME_WINDOW_HOURS - delta_hours) / MAX_TIME_WINDOW_HOURS) * 5.0
    action_impact_score = min(action_count * 1.5, 6.0)
    conflict_word_score = min(conflict_count * 1.0, 8.0)
    fire_word_score = min(fire_count * 1.0, 4.0)
    
    if area_present:
        area_score = 3.0 + min(len(unique_hits(news_text, area_search_terms)) * 1.0, 2.0)
    else:
        area_score = 0.0

    keyword_hit_count = action_count + conflict_count + fire_count + (1 if area_present else 0)
    high_confidence_match = action_count >= MIN_ACTION_HITS and conflict_count >= MIN_CONFLICT_HITS

    if high_confidence_match:
        total_score = 1.0 + time_score + action_impact_score + conflict_word_score - fire_word_score + area_score
    else:
        total_score = 1.0 + (time_score * 0.5) + (action_impact_score * 0.25) + (conflict_word_score * 0.25) - fire_word_score + area_score

    total_score = max(0.0, total_score)

    return {
        "nearest_delta_time_hours": round(delta_hours, 3),
        "time_score": round(time_score, 3),
        "action_impact_score": round(action_impact_score, 3),
        "conflict_word_score": round(conflict_word_score, 3),
        "fire_word_score": round(fire_word_score, 3),
        "area_score": round(area_score, 3),
        "keyword_hit_count": int(keyword_hit_count),
        "high_confidence_match": bool(high_confidence_match),
        "total_score": round(total_score, 3),
        "matched_news_title": news_row.get("title", ""),
        "nearest_news_source": news_row.get("news source", ""),
    }


def correlate_file(cluster_path, news_corpus):
    events = load_firms_events(cluster_path)
    out_rows = []

    event_columns = list(events.columns)

    for _, event_row in events.iterrows():
        reference_time = reference_time_for_event(event_row)
        window_start = reference_time - pd.Timedelta(hours=MAX_TIME_WINDOW_HOURS)
        window_end = reference_time + pd.Timedelta(hours=MAX_TIME_WINDOW_HOURS)
        candidates = news_corpus[(news_corpus["date"] >= window_start) & (news_corpus["date"] <= window_end)].copy()

        event_data = event_row.to_dict()
        event_data.update({
            "matched_news_count": int(len(candidates)),
            "nearest_news_source": "",
            "nearest_news_title": "",
            "nearest_delta_time_hours": pd.NA,
            "time_score": 0.0,
            "action_impact_score": 0.0,
            "conflict_word_score": 0.0,
            "fire_word_score": 0.0,
            "area_score": 0.0,
            "keyword_hit_count": 0,
            "high_confidence_match": False,
            "total_score": 0.0,
        })

        if candidates.empty:
            out_rows.append(event_data)
            continue

        country_name = event_row.get("centroid_country", "")
        area_name = event_row.get("centroid_area_label", "")

        candidates = candidates.copy()
        candidates["delta_hours"] = (candidates["date"] - reference_time).abs().dt.total_seconds() / 3600.0
        candidates = candidates.sort_values(["delta_hours", "date", "title"]).reset_index(drop=True)

        best = None
        for _, news_row in candidates.iterrows():
            score = score_candidate(reference_time, news_row, country_name, area_name)
            if score is not None:
                best = score
                best_news = news_row
                break

        if best is not None:
            event_data.update({
                "nearest_news_source": best["nearest_news_source"],
                "nearest_news_title": best["matched_news_title"],
                "nearest_delta_time_hours": best["nearest_delta_time_hours"],
                "time_score": best["time_score"],
                "action_impact_score": best["action_impact_score"],
                "conflict_word_score": best["conflict_word_score"],
                "fire_word_score": best["fire_word_score"],
                "area_score": best["area_score"],
                "keyword_hit_count": best["keyword_hit_count"],
                "high_confidence_match": best["high_confidence_match"],
                "total_score": best["total_score"],
            })

        out_rows.append(event_data)

    result = pd.DataFrame(out_rows)
    ordered_columns = event_columns + [col for col in EXTRA_COLUMNS if col not in event_columns]
    result = result.reindex(columns=ordered_columns)
    if not result.empty:
        result = result.sort_values(["earliest_detection", "total_score"], ascending=[True, False]).reset_index(drop=True)
    return result


news_corpus = load_news_corpus(NEWS_PROCESSED)
cluster_files = sorted(FIRMS_PROCESSED.glob("clstrd_*.csv"))
if not cluster_files:
    raise FileNotFoundError(f"No clstrd_*.csv files were found in {FIRMS_PROCESSED}")

summary_rows = []
for cluster_path in cluster_files:
    correlated = correlate_file(cluster_path, news_corpus)
    output_path = cluster_path.with_name(cluster_path.name.replace("clstrd_", "crrltd_", 1))
    correlated.to_csv(output_path, index=False)
    summary_rows.append(
        {
            "source_file": cluster_path.name,
            "output_file": output_path.name,
            "input_rows": len(pd.read_csv(cluster_path)),
            "output_rows": len(correlated),
            "matched_rows": int(correlated["matched_news_count"].fillna(0).astype(int).gt(0).sum()) if not correlated.empty else 0,
        }
    )

summary = pd.DataFrame(summary_rows)
summary


# Thermal-News Correlation Engine & Scoring System

This notebook correlates spatio-temporally clustered satellite thermal events (from FIRMS) with a processed news corpus to identify which thermal events are linked to active conflicts or localized industrial/war action, while distinguishing them from natural or accidental wildfires.

---

## 1. Workflow Overview

1. **Load Clustered Satellite Events**: Ingests `clstrd_*.csv` files, which contain aggregated fire event centroids, counts, sensor types (VIIRS/MODIS), and administrative areas (GADM `centroid_country` and `centroid_area_label`).
2. **Load News Corpus**: Merges all `*_all.csv` news databases, parsed with a timezone-aware UTC datetime.
3. **Temporal Window Search**: For each clustered thermal event, a reference time is calculated (midpoint of earliest and latest detection). The engine searches for news reports published within a **$\pm$24-hour temporal window** (`MAX_TIME_WINDOW_HOURS = 24`).
4. **Strict Location Constraints**: To avoid cross-national matches (e.g., Turkey matching Ukrainian news), a candidate news article must explicitly mention:
   - The **Country** name (supporting `Turkey`/`Turkiye` and `Ukraine`/`Ukrainian` variations).
   - **OR** the specific **Area/City** (`centroid_area_label`).
   If neither matches, the candidate is discarded.
5. **Multi-Criteria Scoring**: Candidate matches are scored based on proximity in time, thematic terms, and local geographical alignment.
6. **Data Output & Reordering**: The nearest matched news report with the highest score is attached to the fire cluster. Non-matching events are preserved in the dataset with `0.0` scores. Unnecessary columns (`nearest_news_date`, `nearest_news_link`, and `nearest_news_file`) are pruned, and `nearest_news_source` and `nearest_news_title` are reordered to the very end of the record.

---

## 2. Scoring Model & Multi-Category Keywords

To identify conflict-related fires while penalizing typical/accidental forest fires, text from the news title and source is analyzed against three primary term categories:

### A. Action & Impact Terms (`ACTION_IMPACT_TERMS`)
* **Keywords**: *attack, strikes, explosion, blast, bomb, shelling, bombardment, fire, etc.*
* **Scoring**: $1.5$ points per unique keyword hit, capped at a maximum of **6.0 points** (`action_impact_score`).
* **Purpose**: Identifies active physical violence or sudden thermal impact events.

### B. Conflict Terms (`CONFLICT_TERMS`)
* **Keywords**: *war, missile, rocket, drone, artillery, invasion, airstrike, frontline, combat, etc.*
* **Scoring**: $1.0$ point per unique keyword hit, capped at a maximum of **8.0 points** (`conflict_word_score`).
* **Purpose**: Verifies that the context surrounding the fire is military or geopolitical conflict.

### C. Wildfire & Accidental Fire Terms (`FIRE_TERMS`) - *Minus Criteria*
* **Keywords**: *wildfire, forest fire, brush fire, blaze, inferno, arson, smoke, spread, wind, burn, burned, etc.*
* **Scoring**: $1.0$ point per unique keyword hit, capped at a maximum of **4.0 points** (`fire_word_score`).
* **Purpose**: Used as a **penalty subtraction** in the total score to filter out typical, natural, or accidental seasonal fires.

### D. Temporal Proximity (`time_score`)
* Linear decay over the 24-hour window:
  $$\text{time\_score} = \max\left(0, \frac{24 - |\Delta t_{\text{hours}}|}{24}\right) \times 5.0$$
* Capped at **5.0 points**.

### E. Specific Location Boost (`area_score`)
* If the news explicitly mentions the exact region/city name (`centroid_area_label`), it receives a strong **extra credit baseline of 3.0 points**, plus $1.0$ point per extra unique word hit, capped at **5.0 points**.
* If the specific area is not mentioned, the area score is $0.0$.

---

## 3. Total Score & Confidence Rules

Matches are categorized based on keyword hits to distinguish high-confidence conflict occurrences:

* **High-Confidence Match Criteria**:
  $$\text{Action Hits} \ge 1 \quad \text{and} \quad \text{Conflict Hits} \ge 3$$
  - **High-Confidence Score Formula**:
    $$\text{total\_score} = \max\left(0.0,\, 1.0 + \text{time\_score} + \text{action\_score} + \text{conflict\_score} - \text{fire\_score} + \text{area\_score}\right)$$
  
* **Low-Confidence Score Formula**:
  - The temporal and conflict signals are heavily discounted to prevent false positives:
    $$\text{total\_score} = \max\left(0.0,\, 1.0 + (0.5 \times \text{time\_score}) + (0.25 \times \text{action\_score}) + (0.25 \times \text{conflict\_score}) - \text{fire\_score} + \text{area\_score}\right)$$

---

## 4. Map Visualization Styles

When rendering the interactive Folium maps in `outputs/maps/`:
- **Marker Size**: Scaled dynamically based on `total_score`:
  $$\text{radius} = \text{clamp}\left(3,\, 3 + \frac{\text{total\_score}}{2},\, 15\right)$$
- **Marker Color**:
  - <span style="color:red">**Red**</span>: `high_confidence_match` is `True`.
  - <span style="color:orange">**Orange**</span>: `total_score >= 10.0` but not a high-confidence match.
  - <span style="color:blue">**Blue**</span>: General matches (`total_score < 10.0`).

In [ ]:
import folium
from pathlib import Path
import pandas as pd
import json
# ROOT, FIRMS_DIR, GADM_DIR are set in Cell 3
FIRMS_PROCESSED = FIRMS_DIR / 'data' / 'processed'
OUT_MAP_DIR = FIRMS_DIR / 'outputs' / 'maps'
OUT_MAP_DIR.mkdir(parents=True, exist_ok=True)
# GADM_DIR already set in Cell 3
# helper: safe float cast
def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default
# load GADM geopackages (if geopandas present) and produce GeoJSON features
gadm_layers = []
try:
    import geopandas as gpd
    for gpkg in sorted(GADM_DIR.glob('*.gpkg')):
        try:
            gdf = gpd.read_file(gpkg)
            if not gdf.empty:
                # simplify geometry a little for faster rendering (optional)
                gadm_layers.append((gpkg.stem, json.loads(gdf.to_json())))
        except Exception as e:
            print(f'Warning loading GADM {gpkg}: {e}')
except Exception:
    # geopandas not available; skip GADM overlay
    gadm_layers = []
for csv_path in sorted(FIRMS_PROCESSED.glob('crrltd_*.csv')):
    df = pd.read_csv(csv_path)
    outname = csv_path.stem + '_map.html'
    outpath = OUT_MAP_DIR / outname
    if df.empty or 'centroid_lat' not in df.columns or 'centroid_lon' not in df.columns:
        m = folium.Map(location=[20, 20], zoom_start=2, tiles='CartoDB positron')
        m.save(outpath)
        print(f'Saved correlated map: {outpath} | rows: 0')
        continue
    try:
        center_lat = float(pd.to_numeric(df['centroid_lat'], errors='coerce').mean())
        center_lon = float(pd.to_numeric(df['centroid_lon'], errors='coerce').mean())
    except Exception:
        center_lat, center_lon = 20, 20
    m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles='CartoDB positron')
    # add GADM overlays if available
    for layer_name, geojson in gadm_layers:
        folium.GeoJson(geojson, name=f'GADM: {layer_name}',
                       style_function=lambda f: {"color": "#444444", "weight": 1, "fillOpacity": 0.0}).add_to(m)
    # plot raw points (no clustering) with styling based on scores
    for _, row in df.iterrows():
        lat = _safe_float(row.get('centroid_lat', None))
        lon = _safe_float(row.get('centroid_lon', None))
        if lat is None or lon is None:
            continue
        total_score = float(row.get('total_score', 0.0) or 0.0)
        high_conf = bool(row.get('high_confidence_match', False))
        matched_count = int(row.get('matched_news_count', 0) or 0)
        delta_hours = row.get('nearest_delta_time_hours', '')
        time_score = row.get('time_score', '')
        source = row.get('nearest_news_source', '')
        title = row.get('nearest_news_title', '')
        area = row.get('centroid_area_label', '')
        country = row.get('centroid_country', '')
        
        act_score = row.get('action_impact_score', 0.0)
        conf_score = row.get('conflict_word_score', 0.0)
        fire_score = row.get('fire_word_score', 0.0)
        ar_score = row.get('area_score', 0.0)
        latest_detection = row.get('latest_detection', '')
        average_confidence = row.get('average_confidence_rate', '')
        max_frp = row.get('max_frp', '')
        sum_frp = row.get('sum_frp', '')
        # radius and color mapping
        radius = max(3, min(15, 3 + (total_score / 2.0)))
        if high_conf:
            color = 'red'
        elif total_score >= 10:
            color = 'orange'
        else:
            color = 'blue'
        popup_html = (
            f"<b>{area}, {country}</b><br>"
            f"matched: {matched_count}<br>"
            f"total_score: {total_score} (time: {time_score}, act: {act_score}, conf: {conf_score}, fire: -{fire_score}, area: {ar_score})<br>"
            f"latest_detection: {latest_detection}<br>"
            f"delta_hours: {delta_hours}<br>"
            f"average_confidence: {average_confidence}<br>"
            f"max_frp: {max_frp}<br>"
            f"sum_frp: {sum_frp}<br>"
            f"source: {source}<br>"
            f"title: {title}"
        )
        folium.CircleMarker(location=[lat, lon], radius=radius, color=color, fill=True, fill_opacity=0.7,
                            popup=folium.Popup(popup_html, max_width=400)).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(outpath)
    print(f'Saved correlated map: {outpath} | rows: {len(df)}')

## ML System: Predictive Classification of Thermal Attacks

This section implements a Machine Learning subsystem to predict whether a newly detected satellite thermal event is **War-Related (Missile, Shelling, or Drone Attack)** versus a **Natural/Accidental Wildfire** using *only* physical and spatio-temporal FIRMS attributes. No matching news corpus is utilized during classification inference.

### 1. Label Definition & Ground Truth
We define a binary target variable ($y$) using the historical correlation score:
* **Class 1 (War-Related Attack)**: High-confidence correlation match (`high_confidence_match == True`).
* **Class 0 (Natural/Accidental Fire)**: Uncorrelated or typical seasonal fires (`high_confidence_match == False`).

### 2. Feature Selection ($X$)
Only satellite physical attributes are used:
* `n_points`: Number of fire pixels clustered.
* `max_brightness`: Maximum MODIS brightness temperature (Kelvin).
* `maxbrightness_ti4`: Maximum VIIRS I-4 band brightness temperature (Kelvin).
* `sum_frp`: Sum of Fire Radiative Power (MW).
* `max_frp`: Peak Fire Radiative Power recorded (MW).
* `duration_hours`: Spatio-temporal event duration.

### 3. Machine Learning Models
* **Logistic Regression**: Serves as the linear baseline, trained on standardized features.
* **Decision Tree Classifier**: Trained with a depth ceiling (`max_depth=4`) to prevent overfitting and extract clear, human-interpretable threshold rules.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
# ROOT, FIRMS_DIR are set in Cell 3
FIRMS_PROCESSED = FIRMS_DIR / "data" / "processed"
# Load both datasets
dfs = []
for csv_path in sorted(FIRMS_PROCESSED.glob("crrltd_*.csv")):
    df_temp = pd.read_csv(csv_path)
    if not df_temp.empty:
        dfs.append(df_temp)
if not dfs:
    raise FileNotFoundError("No crrltd_*.csv files were found.")
df = pd.concat(dfs, ignore_index=True)
# Select physical features (only satellite data, no news data!)
feature_cols = [
    "n_points",
    "max_brightness",
    "maxbrightness_ti4",
    "sum_frp",
    "max_frp",
    "duration_hours"
]
# Select features and fill NaNs with 0 (representing absence of that sensor's signal)
X = df[feature_cols].copy()
X["max_brightness"] = X["max_brightness"].fillna(0)
X["maxbrightness_ti4"] = X["maxbrightness_ti4"].fillna(0)
X["sum_frp"] = X["sum_frp"].fillna(0)
X["max_frp"] = X["max_frp"].fillna(0)
X["duration_hours"] = X["duration_hours"].fillna(0)
# Target label: y = 1 for high confidence war match, y = 0 otherwise
y = df["high_confidence_match"].astype(int)
# Train-Test Split (80% train, 20% test, stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Total training samples: {len(X_train)} (Positive: {sum(y_train)}, Negative: {len(y_train) - sum(y_train)})")
print(f"Total testing samples: {len(X_test)} (Positive: {sum(y_test)}, Negative: {len(y_test) - sum(y_test)})")
print("-" * 60)
# --- 1. Logistic Regression Model ---
print("Training Logistic Regression Model...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
print("\\n[Logistic Regression Performance]")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(classification_report(y_test, lr_preds, target_names=["Natural/Other", "War-Related"]))
print("-" * 60)
# --- 2. Decision Tree Model ---
print("Training Decision Tree Model...")
# Capped at depth 4 to prevent overfitting and ensure high interpretability
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
print("\\n[Decision Tree Performance]")
print(f"Accuracy: {dt_model.score(X_test, y_test):.4f}")
print(classification_report(y_test, dt_preds, target_names=["Natural/Other", "War-Related"]))
print("-" * 60)
# Print Feature Importances
print("Decision Tree Feature Importances:")
importances = dt_model.feature_importances_
for name, importance in sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True):
    print(f"  {name:25}: {importance:.4f}")
print("-" * 60)
# Print Textual Decision Tree Rules
print("Learned Decision Rules (Decision Tree):")
tree_rules = export_text(dt_model, feature_names=feature_cols)
print(tree_rules)
print("-" * 60)
# --- 3. Visualizations ---
print("Generating model visualizations...")
# 3.1. Side-by-Side Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, lr_preds, 
    display_labels=["Natural/Other", "War-Related"], 
    cmap="Blues", 
    ax=axes[0]
)
axes[0].set_title("Logistic Regression Confusion Matrix")
ConfusionMatrixDisplay.from_predictions(
    y_test, dt_preds, 
    display_labels=["Natural/Other", "War-Related"], 
    cmap="Oranges", 
    ax=axes[1]
)
axes[1].set_title("Decision Tree Confusion Matrix")
plt.tight_layout()
plt.show()
# 3.2. Logistic Regression Feature Coefficients
plt.figure(figsize=(10, 5))
coef_series = pd.Series(lr_model.coef_[0], index=feature_cols).sort_values()
colors = ["#f87171" if x < 0 else "#60a5fa" for x in coef_series.values]
coef_series.plot(kind="barh", color=colors)
plt.axvline(0, color="gray", linestyle="--", linewidth=1.0)
plt.title("Logistic Regression Feature Coefficients\\n(Positive = Push to War Attack, Negative = Push to Natural Fire)")
plt.xlabel("Coefficient Value (Feature Weight)")
plt.ylabel("Physical Feature")
plt.tight_layout()
plt.show()
# 3.3. Decision Tree Architecture Plot
plt.figure(figsize=(24, 12))
plot_tree(
    dt_model, 
    feature_names=feature_cols, 
    class_names=["Natural/Other", "War-Related"], 
    filled=True, 
    rounded=True, 
    fontsize=10
)
plt.title("Trained Decision Tree Classifier (Max Depth: 4)")
plt.tight_layout()
plt.show()

# Task 4: Dashboard, Insights & Reflection